# Mapping Elevation Tiles with Different Zoom Levels and Sizes

This notebook aims to explain the process and techniques involved in mapping elevation tiles with layers of different zoom levels and sizes. 

## Overview of the Problem Space and Its Importance

The task of mapping image layers, such as satellite imagery, onto elevation grids is a foundational aspect of creating detailed and accurate 3D representations of Planet's surface. This process is crucial in a wide range of applications, from geographic information systems (GIS) to computer vision and 3D scene rendering. The core challenge in this domain arises from the need to balance the level of detail (LOD) in the imagery with performance and resource constraints, especially in dynamic environments where the viewer's perspective constantly changes.

### High LOD in Satellite Imagery

Layers, such Satellite imagery often provides a high level of detail, capturing the Body's surface with precision. When mapping these images onto elevation grids, a high LOD ensures that the resulting 3D terrain model is both accurate and visually compelling. This high-definition mapping is essential for tasks that require precise spatial data.

### The Challenge of Dynamic Perspectives

However, the necessity for high-resolution imagery presents a significant challenge in applications involving 3D scene rendering, particularly in computer vision. As the viewer's perspective changes, not all parts of the terrain require the same level of detail. Distant terrain features can be represented with lower-resolution imagery without compromising the overall quality of the scene, reducing the computational load and improving rendering performance.

### A Dynamic Approach to Image Layer Mapping

To address this issue, a dynamic approach to mapping image layers onto elevation grids is essential. This approach must adjust the LOD of imagery in real-time, based on the viewer's distance from different parts of the terrain. Such a strategy requires sophisticated logic at the GPU level, where a single elevation grid model can be manipulated alongside a stack of image layers and textures. This approach optimizes resources by reducing the number of draw calls and efficiently managing texture memory.

### Leveraging WebGL2 and WebGPU

Implementing this dynamic mapping system necessitates advanced GPU programming techniques, utilizing the capabilities of WebGL2 and the emerging WebGPU standard. These technologies provide the necessary tools for adjusting LOD dynamically, allowing for real-time optimizations that can significantly enhance the efficiency and performance of 3D rendering applications.

## Problem Inputs and Their Implications

The primary inputs to the problem of mapping image layers onto elevation grids involve:

1. **An Elevation Tile System**: This system is characterized by metrics such as `elevationTileSize`, which defines the dimensions of each tile in the system. These tiles are the basic units used to construct a detailed elevation map of the body's surface.

2. **A Consequent Grid as Mesh**: Accompanying and resulting from the elevation tile system is a mesh grid with dimensions of `ew=eh=tileSize+1`. This specific dimensioning is crucial for the construction and rendering of the elevation grid in a 3D space.

![Elevation Grid](../images/elevation_grid.png "the elevation grid reference")


### The Significance of the "+1" in Mesh Dimensions

The addition of `+1` to the mesh grid dimensions (`ew` and `eh`) is a critical aspect of this system, designed to address a common challenge in tiling systems: the seamless connection of adjacent tiles. 
In a tiled elevation grid, each tile represents a specific portion of the body's surface. However, without an overlap or a mechanism for connection, there would be visible gaps between adjacent tiles due to the discrete nature of the data and potential rounding errors in rendering. The "+1" in the dimensions of the mesh grid effectively creates an overlap between adjacent tiles, ensuring that there are no gaps in the rendered 3D terrain. This overlap allows for the vertices at the edge of one tile to seamlessly connect with the corresponding vertices of neighboring tiles, facilitating a continuous and cohesive mapping of image layers across the entire elevation grid.

In [1]:
elevationTileSize = 256
ew = elevationTileSize + 1
hw = elevationTileSize + 1
elevationLOD = 1

Depending on the desired definition and size of the map, different strategies can be employed for mapping image layers onto the elevation. Mapping can be executed using several image tiles at a higher Level of Detail (LOD), or it might involve using a lower definition, which leads to utilizing only a portion of an image.

<p align="left">
  <img src="../images/map_greater_lod.png" alt="mapping higher level of detail" style="width: auto; max-width: 48%; margin-right: 2%; vertical-align: top;">
  <img src="../images/map_lower_lod.png" alt="mapping lower level of detail" style="width: auto; max-width: 48%; vertical-align: top;">
</p>

Then we define the additional variables.<br>
<br>
⚠️ **Warning:** At this point, we assume (this is optimistic) that the tile metrics are consistent between the elevation and the mapping. This is why we can use the LOD (Level of Detail) difference to compute the varying scale and offset. However, to introduce broader support, we need access to the metrics on both tile systems and compare either geographic coverage or projected XY values - also assuming that the projections are compatible


In [2]:
import math
mapTileSize = 256
mw = mapTileSize
mh = mapTileSize
mapLOD = 3

d = mapLOD - elevationLOD 
ad = abs(d)
if d > 0:
    s = 1 << ad
elif d < 0:
    s = 1.0 / float(1 << ad)
else:
    s = 1.0
n = math.ceil(s)
print(f"d={d}, s={s}, n={n}")

d=2, s=4, n=4


By utilizing these variables defined at the CPU level, the GPU can dynamically compute the UV map coordinates using the following formulas:
$$\text u\_map = uOffset + fract(\text u\times \text s)$$
$$\text v\_map = vOffset + fract(\text v\times \text s)$$

where

* $s$ is the scale
* $u,v$ are one of the UV's coordinate of the grid
* The $fract()$ function returns the fractional part of a floating point number, essentially discarding the integer part.
* $uOffset$ and $vOffset$ are the offsets added to define the origin of the mapping.

In [3]:
def fract(number):
    """
    Returns the fractional part of the given number.
    
    Parameters:
    - number (float): The input number.
    
    Returns:
    - float: The fractional part of the input number.
    """
    return number - int(number)

u = 0.249 # this is var input
v = 0.126 # this is var input
us = u*s
vs = v*s
u_map = fract(us)
v_map = fract(vs)

The variable `n` specifies the dimensions of a buffer, which is $[n, n]$. This buffer stores references to an image within a 2D/3D texture sampler. Consequently, the indices within this buffer are determined as follows:
$$i = floor(\text u\times \text s)$$
$$j= floor(\text v\times \text s)$$
$$0 \leq i < n, \quad i \in \mathbb{Z}_{\geq 0}$$
$$0 \leq j < n, \quad j \in \mathbb{Z}_{\geq 0}$$

The indices \(i, j\) are utilized to retrieve the texture index within the 3D sampler. This index is assigned at the CPU level, specifically within instance variables.

In [4]:
i = math.floor(us)
j = math.floor(vs)

print(f"u_map={u_map}, v_map={v_map}, i_map={i}, j_map={j}")

u_map=0.996, v_map=0.504, i_map=0, j_map=0


In the configuration described above, `uOffset` and `vOffset` are implicitly set to 0. However, when the map's Level Of Detail (LOD) is lower than that of the elevation LOD, it becomes crucial to accurately adjust these values to accurately represent the grid's position within the texture. 

While we have assumed that the metrics are fully compatible between elevation and texture, there are scenarios where this is not the case (for example, different tile sizes). In such instances, it might be necessary to define the offsets by obtaining the actual geographic coordinates of both tiles and expressing the offset as a normalized position of the Elevation Tile within the Mapping Tile.

## Pratical Shader Implementation.

Using a NxN matrix to index the texture layer for each instance is not straightforward when programming shaders. Shaders are limited in the types of data they can handle, and they do not support arrays, especially dynamic ones. Moreover, the typical approach for handling per-instance data involves storing all necessary information into a Buffer Object. This data is then accessed in the shader using the instance ID to index into the buffer. This method complicates the process and is less direct than simpler data handling techniques in shader programming.
So, we may think about other alternatives.

## Access the Geographic Coordinates

According to the standard Web-Mercator projection, using WGS84 ellipsoid and EPSG:3857, the formulas are the following
Given TX and TY with LOD

$$a = 2^{\text{{levelOfDetail}}}$$
$$\text{lon} = -180 + \left(\frac{x}{a}\right) \times 360$$
$$b = \pi - \left(\frac{2 \pi y}{a}\right)$$
$$\text{lat} = \left(\frac{180}{\pi}\right) \times \text{atan}\left(0.5 \times \left(e^b - e^{-b}\right)\right) = \left(\frac{180}{\pi}\right) \times \text{{atan}}\left(\text{sinh}\left(b\right)\right)$$

The expression 
$$e^b - e^{-b}$$ 
can be simplified using the hyperbolic sine function, denoted as 
$$\text{sinh}\left(b\right)$$
The hyperbolic sine of n is defined as:

$$\text{sinh}\left(b\right) = \frac{e^n - e^{-n}}{2}$$

When it comes to implementing the calculation in code, especially if you're concerned with performance or readability, using a direct function call to compute the hyperbolic sine is usually the fastest and most straightforward method, assuming your programming language provides one. Whenever possible, prefer using built-in mathematical functions provided by your programming language's standard library or math libraries. These functions are typically optimized for performance and numerical stability, offering more reliable and faster computations than manual implementations of the same formulas.
However, keep in mind that some trigonometric implementations, such as those found in GPUs, may suffer from poor accuracy. In such cases, you might prefer the expanded version with exponential notation.

In [5]:
class Cartesian2:

    @staticmethod
    def Zero() -> "Cartesian2": 
        return Cartesian2(0, 0)

    def __init__(self, x: float = 0 , y: float = 0):
        self.x = x
        self.y = y

    def __str__(self) -> str:
        return f"({self.x}, {self.y})"


In [6]:
class Geo2:
    
    @staticmethod
    def Zero() -> "Geo2":
        return Geo2(0, 0)
        
    def __init__(self, lat: float = 0 , lon: float = 0):
        self.lat = lat
        self.lon = lon

    def __str__(self) -> str:
        return f"({self.lat}, {self.lon})"


In [7]:
def getTileXYToLatLon(x:float, y:float, levelOfDetail:int):
    a = 2 ** levelOfDetail
    lon = -180 + (x / a) * 360
    b = math.pi - (math.pi * 2 * y) / a
    lat = (180.0 / math.pi) * math.atan(math.sinh(b))
    return Geo2(lat, lon)

Having discussed this, it's also worth mentioning that when dealing with Spherical Coordinates, the situation in tile projections used for web mapping, such as the widely-utilized Web Mercator projection, becomes more complex. Specifically, the latitude of the tiles is not evenly distributed across the range from -90 degrees to +90 degrees. This uneven distribution arises because the Web Mercator projection and similar systems employ a spherical or ellipsoidal model of the Earth. They apply mathematical functions to project the Earth's surface onto a flat map, facilitating easy zooming and panning.

Conversely, the handling of longitude in tile projections like the Web Mercator projection differs significantly from that of latitude. The distribution of longitude lines (meridians) is uniform across the globe. In the Web Mercator projection, each tile at a specific zoom level spans the same range of longitudes, regardless of its position on the map. This uniform distribution of longitude is a defining characteristic of cylindrical map projections, including the Mercator projection.

In [8]:
# Level of Detail
lod = mapLOD
r = (2 ** lod) +1

# Generate the 2D array for x and y 
lat_lon_2d_array = [[getTileXYToLatLon(x, y, lod) for x in range(r)] for y in range(r)]

# Print the 2D array
for row in lat_lon_2d_array:
    for a in row:
        print(f"({a.lat:7.2f}, {a.lon:7.2f})", end=' ')
    print()  # New line after each row

(  85.05, -180.00) (  85.05, -135.00) (  85.05,  -90.00) (  85.05,  -45.00) (  85.05,    0.00) (  85.05,   45.00) (  85.05,   90.00) (  85.05,  135.00) (  85.05,  180.00) 
(  79.17, -180.00) (  79.17, -135.00) (  79.17,  -90.00) (  79.17,  -45.00) (  79.17,    0.00) (  79.17,   45.00) (  79.17,   90.00) (  79.17,  135.00) (  79.17,  180.00) 
(  66.51, -180.00) (  66.51, -135.00) (  66.51,  -90.00) (  66.51,  -45.00) (  66.51,    0.00) (  66.51,   45.00) (  66.51,   90.00) (  66.51,  135.00) (  66.51,  180.00) 
(  40.98, -180.00) (  40.98, -135.00) (  40.98,  -90.00) (  40.98,  -45.00) (  40.98,    0.00) (  40.98,   45.00) (  40.98,   90.00) (  40.98,  135.00) (  40.98,  180.00) 
(   0.00, -180.00) (   0.00, -135.00) (   0.00,  -90.00) (   0.00,  -45.00) (   0.00,    0.00) (   0.00,   45.00) (   0.00,   90.00) (   0.00,  135.00) (   0.00,  180.00) 
( -40.98, -180.00) ( -40.98, -135.00) ( -40.98,  -90.00) ( -40.98,  -45.00) ( -40.98,    0.00) ( -40.98,   45.00) ( -40.98,   90.00) ( -40.9

 As we endeavor to map UV coordinates onto a mesh, the irregular distribution complicates matters. We can dynamically define our UV coordinates based on the geographic coordinates of the texture. This strategy undermines our initial approach of storing pre-computed UV coordinates in the grid and sharing them across instances. Moreover, performance considerations must be taken into account, especially in the Pixel Shader, where we aim to avoid complex calculations.

Theoretically, when dealing with a 3D vertex in our mesh, it corresponds to a specific geographic coordinate. The distribution of all vertices is determined by the spatial context in which we choose to work. The question then arises: do we apply elevation data onto an ellipsoidal surface, adhering to the Earth's actual shape, or do we retain the vertices in a projected space, such as that defined by the Web Mercator projection?

When you're constructing a 3D model that represents geographical features, each vertex in your mesh represents a point on the Earth's surface. The position of these vertices—and therefore, the accuracy and realism of your model—depends greatly on how you choose to interpret their geographic coordinates:

- **Ellipsoidal Surface**: If you opt to map your vertices onto an ellipsoidal surface, you're considering the Earth's curvature and modeling your features in a geodetically accurate manner. This approach is more complex computationally but results in a more accurate representation of the Earth's surface, especially for applications like flight simulation, astronomy, and certain types of geospatial analysis.

- **Projected Space (e.g., Web Mercator)**: Alternatively, keeping your vertices in a projected space simplifies the math, especially for data visualization and web mapping. However, this comes at the cost of accuracy, particularly in terms of area and distance distortion, which becomes more pronounced as you move away from the equator. Web Mercator is a popular choice for online maps because it maintains angles and shapes (making it conformal), facilitating easier navigation and visualization. However, it significantly distorts scale and area, especially near the poles.<br>

The choice between using an ellipsoidal model or a projected model for your 3D vertices depends on the specific needs of your application:

- **For high accuracy and representation of physical phenomena** (e.g., simulating satellite orbits or geological surveys), an ellipsoidal model is preferable.
  
- **For general mapping, visualization, and web applications**, a projected model like Web Mercator might be sufficient and more practical.

## Ellipsoidal Surface

This is ultimately how we aim to represent the surfaces of bodies: by using tangent space for the calculations and employing both ellipsoidal coordinates and planet-centric Cartesian coordinates to convert to local tangent plane coordinates.

## Projected Space

Finally, this leads to the simplest method for calculating UV coordinates. The process involves transitioning from the UV coordinates of the mesh to the UV coordinates of the map, taking into account that the map's Level Of Detail (LOD) and size may differ. 
Therefore, we might conclude that the previously mentioned logic is not entirely accurate and seek a more effective method for calculating the scale `s`, rather than relying solely on the differences in Level Of Detail (LOD).

Lets first state that we are working into the UV Space of the projected Map or texture, which is implies we are working into "pixel" coordinate where the specific texture is a portion, starting at x0,y0 with a size of w and h.
Given a lat,Lon this is simple to compute the x,y coordinate into this texture space.
$$-85.05112878 \leq \text lat \leq 85.05112878$$
$$-180 \leq \text lon \leq 180$$
$$ \text x = \frac{\text lon + 180}{360}$$
$$ \text sinLatitude = \text{sin}( \text lat \times \frac{\pi}{180})$$
$$\text y = 0.5 - \frac{\log\left(\frac{1 + \text{sinLatitude}}{1 - \text{sinLatitude}}\right)}{4 \pi}$$
$$\text{mapSize} = \text{tileSize} \times 2^{\text{LOD}}$$
$$\text x = \text{round}\left(\max\left(0, \min\left(x \times \text{mapSize} + 0.5, \text{mapSize} - 1\right)\right)\right)$$
$$\text y = \text{round}\left(\max\left(0, \min\left(y \times \text{mapSize} + 0.5, \text{mapSize} - 1\right)\right)\right)$$

The reason for adding 0.5 before casting to an integer is to round the floating-point value to the nearest integer. Without this step, the cast to an integer would truncate the fractional component of the floating-point value towards zero.This rounding step is necessary to ensure that the resulting point coordinates are centered on the cell, as required by the cell-centered convention.<br>
That said, the metric and formulas used in tile servers limit the latitude to +/- 85.05112878 degrees, which corresponds to the use of a Tangent operation in the following formula:
$$
\text{latitude} = 90 - \left(\frac{360 \times \arctan\left(\exp\left(-y \times 2 \times \pi\right)\right)}{\pi}\right)
$$



In [9]:
def clamp(value, min_value, max_value):
        return max(min_value, min(value, max_value))

def getLatLonToPointXY(latitude, longitude, levelOfDetail, tileSize):
    x = (longitude + 180) / 360
    sinLatitude = math.sin(latitude * math.pi / 180)
    y = 0.5 - math.log((1 + sinLatitude) / (1 - sinLatitude)) / (4 * math.pi)
    mapSize = tileSize << levelOfDetail
    return Cartesian2(int(round(clamp(x * mapSize + 0.5, 0, mapSize - 1))), int(round(clamp(y * mapSize + 0.5, 0, mapSize - 1))))

def getTileXYToPointXY(tileX, tileY, tileSize):
    return Cartesian2(tileX * tileSize, tileY * tileSize)

Given a map size of 512, LOD map of 1, and Elevation LOD of 2

In [10]:
mapTileSize = 512
elevationGeo = getTileXYToLatLon(2,3,2)
print(f"elevationGeo={elevationGeo}")
pointInMapSpace = getLatLonToPointXY(elevationGeo.lat, elevationGeo.lon, 1, mapTileSize)
print(f"pointInMapSpace={pointInMapSpace}")
point0 = getTileXYToPointXY(1, 1, mapTileSize)
point1 = getTileXYToPointXY(2, 2, mapTileSize)
print(f"point0={point0}, point1={point1}")
uOffset = (pointInMapSpace.x-point0.x) / mapTileSize
vOffset = (pointInMapSpace.y-point0.y) / mapTileSize
print(f"uOffset={uOffset}, vOffset={vOffset}")


elevationGeo=(-66.51326044311186, 0.0)
pointInMapSpace=(512, 768)
point0=(512, 512), point1=(1024, 1024)
uOffset=0.0, vOffset=0.5


Given a map size of 256, LOD map of 3, and Elevation LOD of 1

In [11]:
mapTileSize = 256
elevationGeo = getTileXYToLatLon(0,0,1)
print(f"elevationGeo={elevationGeo}")
pointInMapSpace = getLatLonToPointXY(elevationGeo.lat, elevationGeo.lon, 3, mapTileSize)
print(f"pointInMapSpace={pointInMapSpace}")
point0 = getTileXYToPointXY(2, 3, mapTileSize)
point1 = getTileXYToPointXY(2, 4, mapTileSize)
print(f"point0={point0}, point1={point1}")
uOffset = (pointInMapSpace.x-point0.x) / mapTileSize
vOffset = (pointInMapSpace.y-point0.y) / mapTileSize
print(f"uOffset={uOffset}, vOffset={vOffset}")

elevationGeo=(85.0511287798066, -180.0)
pointInMapSpace=(0, 0)
point0=(512, 768), point1=(512, 1024)
uOffset=-2.0, vOffset=-3.0


The question that arises now is how to easily obtain geographical coordinates, given that the latitude, coming from the elevation tile, is not evenly distributed. <br>
Fortunately, we are working in a mapping space where the local distortion will not be noticeable. Therefore, we can calculate the northwest (NW) offset for each mapping tile relative to the grid's NW corner and save this information as an instance buffer in the previously seen `[n,n]` array, which become an array of Vector3 with x,y as offsets and z the index of the texture into the sampler.